Currently, the working version of both the average and max pooling backward passes suffer from major GPU bottleneck issues:

* Max: There are two main issues with the implementation. The first, we broadcast the axes into four seperate 4D arrays taking up four times the memory space. The second issue is that using `cp.add.at`. This method is designed to handle cases where multiple indices point to the exaxct same memory address, but because we are using this atomic operation, the GPU has to introduce a lock at a memory address serilaizing the execution. 

* Average: There are three main issues with the elementwise kernel appraoch. 
    1. We used `atomicAdd(...)` locking a specific memory address such that only a single thread is able to update its space at a time. Average pooling will distribute gradients to every single pixel inside a pooling window. If our windows overlap slightly (e.g., $3 \times 3$ filters with a stride of $2$), multiple threads will simultaneously compute the exact same `target_idx`.

    2. GPUs read data from VRAM in wide, contiguous blocks(with sizes of 32-byte or 128-byte segments). The GPU will use a hardware scheduling unit (AMD Wavefront, or Nvidia Warp) will send a **single memory transaction**, sending all the byte information needed for each thread to read at the same time. For example, if we have 32 threads that would like to read data to perform our average pooling calculation, then we'll send a 128 byte block of data (4 bytes per thread) in a single transaction. **However**, inside the code the line appears:
    ```python
    int target_idx = ((s * H_in + h_in) * W_in + w_in) * C + c;
    ```
    Because the kernel is launched over `dvalues.ravel()`, each thread will handle a different output position. In other words, instead of us cleanly sending a uniform block of memory in a single sweep, the threads are scatering data all over the place, forcing the GPU to repeatedly send fetch instructions.

    3. Misuse of `cp.ElementwiseKernel` for a scatter operation. 
    In this case, `ElementwiseKernel` is best use under the assumption of **element-to-elemnt mapping** (e.g., Thread $i$ reads input $i$ and writes cleanly to output $i$)
    Here, we used `raw float32 dinputs` and wrote a custon kernel to handle the scattering and unpooling operations. However, because we didn't follow the simple rule the elementwise kernel is intended for, we cannot optimize memory block loading or shared cache for loops it doesn't know exists. 

Below is the old pooling layer with its backwards pass:

In [ ]:
import cupy as cp

scatter_avg_pooling_kernel = cp.ElementwiseKernel(
    in_params='''
        float32 dval,
        int32 S, int32 H_out, int32 W_out, int32 C,
        int32 H_in, int32 W_in,
        int32 fH, int32 fW,
        int32 sH, int32 sW
    ''',
    out_params='raw float32 dinputs',
    operation=r'''
        // i is the linear index into dvalues
        // Decode: (s, h_out, w_out, c)
        int c = i % C;
        int w_out = (i / C) % W_out;
        int h_out = (i / (C * W_out)) % H_out;
        int s = i / (C * W_out * H_out);
        
        // Gradient to distribute to each position in the pool
        float grad_per_position = dval / (fH * fW);
        
        // Calculate starting position in input
        int h_start = h_out * sH;
        int w_start = w_out * sW;
        
        // Distribute gradient to all positions in this pool window
        for (int fh = 0; fh < fH; fh++) {
            for (int fw = 0; fw < fW; fw++) {
                int h_in = h_start + fh;
                int w_in = w_start + fw;
                
                // Calculate linear index in dinputs
                int target_idx = ((s * H_in + h_in) * W_in + w_in) * C + c;
                
                // Atomic add to handle overlapping windows
                atomicAdd(&dinputs[target_idx], grad_per_position);
            }
        }
    ''',
    name='scatter_avg_pooling'
)

class Pooling:
    def backward(self, dvalues):
        
        dvalues = dvalues.astype(cp.float32, copy = False)
        #We want the same shape as self.inputs, we'll populate the tensor with zeros at first then unpool later.
        self.dinputs = cp.zeros_like(self.inputs, dtype=cp.float32)
        S, H_out, W_out, C = dvalues.shape
        H_in, W_in = self.inputs.shape[1:3]
        fH, fW = self.filter_size
        sH, sW = self.strides
        
        if self.pooling_type == "max":
            max_rows, max_cols = self.max_indicies
            
            s_idx = cp.arange(S)[:, None, None, None]      # Shape: (S, 1, 1, 1)
            h_idx = cp.arange(H_out)[None, :, None, None]  # Shape: (1, H_out, 1, 1)
            w_idx = cp.arange(W_out)[None, None, :, None]  # Shape: (1, 1, W_out, 1)
            c_idx = cp.arange(C)[None, None, None, :]      # Shape: (1, 1, 1, C)
            
            # Calculate where in the input each gradient should go
            # Broadcasting creates arrays of shape (S, H_out, W_out, C)
            input_h = h_idx * sH + max_rows  # h_idx broadcasts, max_rows is already (S, H_out, W_out, C)
            input_w = w_idx * sW + max_cols
            
            # Accumulate gradients at the right positions
            # cp.add.at handles if multiple output positions map to same input position
            cp.add.at(self.dinputs, (s_idx, input_h, input_w, c_idx), dvalues)
        
        elif self.pooling_type == "average":
            
            scatter_avg_pooling_kernel(
                dvalues.ravel(),
                S, H_out, W_out, C,
                H_in, W_in, 
                fH, fW,
                sH, sW,
                self.dinputs.ravel()
            )
        return self.dinputs

## How do we Optimize our Backwards Pass? 

 Well for one, we'll do away with a custom kernel for our project as we're able to use two routes to perform the backwards pass of our pooling layer. The first is under the assumption that our filter sizes $(fH, fW)$ match our stride height and width $(sH, sW)$. We'll still account for the average and max versions of the code, but we'll divide each approach into two different implementations, where our second approach will be a more generalized version that supports any filter and stride dimensions at the cost of speed. 

Inside `Backpropagation_Pooling`, there includes theory and implementation of both the max and average pooling backward passes. This notebook will be used as a continuation of that notebook, where we'll introduce the idea and implementation of these faster methods. 

## Max Pooling

In max pooling we'll recieve the gradient `dvalues` which is our $\frac{\partial y}{\partial L}$ which is our forward pass outputs values wrt. the loss function. We need to arrive at `dinputs` or $\frac{\partial x}{\partial L}$. An expanded form of our backwards pass is written below:

$$
\nabla_X L_{s, h', w', c} = \sum_{h, w} \nabla_Y L_{s, h, w, c} \cdot \mathbb{I}\left( X_{s, h', w', c} = Y_{s, h, w, c} \right)
$$

where
* $\nabla_X L_{s, h', w', c}$: The `dinputs` or downstream gradient wrt. the loss function. Our batch elements and channels never change during pooling, meaning the layout matches the original input shape $(S, H_{in}, W_{in}, C)$. $h'$ is a specific row index inside the range of `H_in`, and $w`$ is a specific column index inside the range of `W_in`. 

* $\sum {h, w}$: Evaluates a summation across all sliding window spatial coordinates $(h, w)$

* $\nabla_Y L_{s, h, w, c}$: Our `dvalues` or upstream gradient wrt. the loss function. $h$ is a specific row index inside the range of `H_out`, and $w$ is a specific column index inside the range of `W_out`. 

* $\mathbb{I}\left( X_{s, h', w', c} = Y_{s, h, w, c} \right)$: Here, we verify whether the original input pixel value $X$ at position $(h', w')$ matches the pooled maximum output value $Y$ extracted from its local window. When they match, the upstream gradient value is routed entirely through to this single index. 

This last part is the most informative, if we hit a position that contained a max value, then we have a position that influenced the loss function at input pixel $X$. Otherwise, if the condition is false the indicator function evaluates to 0, meaning that we have hit a point that did not influence the position of the loss function. 


## Implementing our Max Pooling Pass: 
